In [1]:
import pandas as pd
import duckdb as ddb
from pathlib import Path

In [5]:
DATAPATH = '/Users/biancachiusano/Desktop/uva/thesis/Thesis/datasets/cloud_energy_consumption/full_nodes_featurestwo.parquet'

In [3]:
con = ddb.connect(':memory:')

In [6]:
# importing the raw node data
# read parquet
con.execute(f"""CREATE TABLE nodes_table AS SELECT * FROM read_parquet('{DATAPATH}')""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
VM_HARDWARE = "/Users/biancachiusano/Desktop/uva/thesis/Thesis/datasets/cloud_energy_consumption/vms/2024-12-14T000000Z_2025-04-13T235959Z/vms.csv"
VM_DATA = "/Users/biancachiusano/Desktop/uva/thesis/Thesis/datasets/cloud_energy_consumption/nodes-vms/2024-12-14T000000Z_2025-04-13T235959Z/**/*.csv"

## VM HARDWARE

In [8]:
# VM HARDWARE 
con.query(f"""CREATE OR REPLACE TABLE vmhardware AS SELECT * FROM '{VM_HARDWARE}'""")
con.query("""SELECT * FROM vmhardware LIMIT 5""").show()

┌──────────┬──────────┬────────────┬───────────┬────────┬───────────┬─────────┬──────────────┐
│  vm_id   │ user_id  │ project_id │ image_ref │ vcpus  │ memory_mb │ root_gb │ ephemeral_gb │
│ varchar  │ varchar  │  varchar   │  varchar  │ double │  double   │ double  │    double    │
├──────────┼──────────┼────────────┼───────────┼────────┼───────────┼─────────┼──────────────┤
│ 6239bcf4 │ 39786247 │ 4ba6ce42   │ 85ecdff4  │   32.0 │   60000.0 │    30.0 │        160.0 │
│ 80458243 │ bd41a7e4 │ 74dfdf00   │ cb92696e  │    1.0 │    2000.0 │    10.0 │         20.0 │
│ 5d9f6237 │ 7190ddfa │ 906147a9   │ 78c5906a  │    4.0 │    7500.0 │    30.0 │         60.0 │
│ 941785dc │ 1a324c91 │ f09054da   │ 23658854  │    2.0 │    4000.0 │    10.0 │        100.0 │
│ fb856e0f │ c1eb1c1c │ f09054da   │ c015d680  │    4.0 │    7500.0 │    30.0 │         60.0 │
└──────────┴──────────┴────────────┴───────────┴────────┴───────────┴─────────┴──────────────┘



## VM DATA

In [9]:
con.execute(f"""
                CREATE OR REPLACE VIEW vm_data AS
                SELECT *
                FROM read_csv_auto('{VM_DATA}',
                                filename=false,
                                union_by_name=true)
            """)

### Merging

In [10]:
con.execute("""
    CREATE OR REPLACE TABLE vm_merged AS
    SELECT 
        d.*, 
        h.*
    FROM vm_data d
    LEFT JOIN vmhardware h
    ON d.vm_id = h.vm_id
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### Imputing missing values
Forward Fill and 0s

In [11]:
con.execute("""
    CREATE OR REPLACE TABLE vm_filled AS
    SELECT *,
        LAST_VALUE(scaphandre_vm_power_total_watts IGNORE NULLS)
        OVER (
            PARTITION BY vm_id 
            ORDER BY timestamp
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS power_filled
    FROM vm_merged
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [12]:
con.execute("""
    CREATE OR REPLACE TABLE vm_final AS
    SELECT *,
        COALESCE(power_filled, 0) AS power_clean
    FROM vm_filled
 """)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [13]:
con.query("""SELECT * FROM vm_final LIMIT 5""").show()

┌──────────────────────────┬──────────┬─────────────────┬──────────────────┬─────────────────────────────────┬──────────┬──────────┬────────────┬───────────┬────────┬───────────┬─────────┬──────────────┬──────────────┬─────────────┐
│        timestamp         │  vm_id   │ hypervisor_name │ hypervisor_group │ scaphandre_vm_power_total_watts │ vm_id_1  │ user_id  │ project_id │ image_ref │ vcpus  │ memory_mb │ root_gb │ ephemeral_gb │ power_filled │ power_clean │
│ timestamp with time zone │ varchar  │     varchar     │     varchar      │             double              │ varchar  │ varchar  │  varchar   │  varchar  │ double │  double   │ double  │    double    │    double    │   double    │
├──────────────────────────┼──────────┼─────────────────┼──────────────────┼─────────────────────────────────┼──────────┼──────────┼────────────┼───────────┼────────┼───────────┼─────────┼──────────────┼──────────────┼─────────────┤
│ 2025-01-16 09:36:00+01   │ 1df98863 │ 5e87c22e        │ a6177608  

In [23]:
#TIMESTAMP = '2025-01-16T08:33:00Z'
TIMESTAMP = '2025-01-16 08:33:00 UTC'

### Get VM list per node 

In [24]:
# Check timestamp notation
con.query(f"""
    SELECT 
        hypervisor_name AS node_name,
        LIST(vm_id) AS vm_ids
    FROM vm_final
    WHERE timestamp = '{TIMESTAMP}'
    GROUP BY hypervisor_name
""").show()

┌───────────┬──────────────────────────────────────────────────────────────────────────────────┐
│ node_name │                                      vm_ids                                      │
│  varchar  │                                    varchar[]                                     │
├───────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ 7c804642  │ [cb521e0f]                                                                       │
│ b654db47  │ [e80b91dd]                                                                       │
│ 0eb86326  │ [ef84013a]                                                                       │
│ c9900ab9  │ [5486492a, 654a92dd]                                                             │
│ 6bb5c698  │ [f545ef7a, 4aeb93ea, be1a8732, b963d466, eae4dca6]                               │
│ ea7f6999  │ [91dc05b7]                                                                       │
│ 6d834fb8  │ [c6b13b89]      

### VM Aggregation - get node load - use for comparison - ALSO nodes_t

In [16]:
con.query(f"""
    SELECT 
        hypervisor_name AS node_name,
        SUM(power_clean) AS total_power,
        COUNT(*) AS vm_count
    FROM vm_final
    WHERE timestamp = '{TIMESTAMP}'
    GROUP BY hypervisor_name
""").show()

┌───────────┬─────────────────────┬──────────┐
│ node_name │     total_power     │ vm_count │
│  varchar  │       double        │  int64   │
├───────────┼─────────────────────┼──────────┤
│ 82415d68  │                0.44 │       11 │
│ d8d6d1e0  │                0.11 │        1 │
│ 6342c79e  │                0.18 │        1 │
│ 7f4f3762  │                0.12 │        6 │
│ 28e8b849  │                0.01 │        1 │
│ 3d7960e7  │  1.1600000000000001 │        7 │
│ 58512373  │                0.25 │        7 │
│ 5c5e400b  │                0.07 │        1 │
│ 5333662e  │              118.55 │        1 │
│ e2b6ed0e  │   9.450000000000001 │        5 │
│    ·      │                  ·  │        · │
│    ·      │                  ·  │        · │
│    ·      │                  ·  │        · │
│ a22bdd7c  │                0.07 │        2 │
│ 7f384201  │ 0.22000000000000003 │        4 │
│ 6ff55332  │                0.24 │        3 │
│ 8d5a6c7e  │                0.41 │        1 │
│ 764b2e98  │

# Timestamps

In [17]:
timestamps = con.execute("""
    SELECT DISTINCT timestamp AT TIME ZONE 'UTC'
    FROM vm_final
    ORDER BY timestamp
""").fetchall()

In [18]:
timestamps[0]

(datetime.datetime(2024, 12, 14, 0, 0),)

In [25]:
for (t,) in timestamps[0:5]:
    print(f"Processing timestamp: {t} UTC")


Processing timestamp: 2024-12-14 00:00:00 UTC
Processing timestamp: 2024-12-14 00:03:00 UTC
Processing timestamp: 2024-12-14 00:06:00 UTC
Processing timestamp: 2024-12-14 00:09:00 UTC
Processing timestamp: 2024-12-14 00:12:00 UTC


In [27]:
LOWER_THRESHOLD = 10
UPPER_THRESHOLD = 90

In [ ]:
 # for each node at time t, extract name, cpu usage, and vm count and vm ids
node_df_t = con.execute("""
                        SELECT 
                            n.node_name,
                            n.cpu_usage_percent,
                            COUNT(v.vm_id) AS vm_count,
                            LIST(v.vm_id) AS vm_ids
                        FROM nodes_table n
                        LEFT JOIN vm_final v
                            ON n.timestamp = v.timestamp
                            AND n.node_name = v.hypervisor_name
                        WHERE n.timestamp = ?
                        GROUP BY n.node_name, n.cpu_usage_percent
                    """, [TIMESTAMP]).df()

print(node_df_t.head())
    
overloaded = node_df_t[node_df_t["cpu_usage_percent"] > UPPER_THRESHOLD]
print("Overloaded nodes:")
print(overloaded)
underloaded = node_df_t[node_df_t["cpu_usage_percent"] < LOWER_THRESHOLD]
print("Underloaded nodes:")
print(underloaded)

In [29]:
node_df_t

,node_name,cpu_usage_percent,vm_count,vm_ids
0,2fded806,0.60,8,"[62ed6b58, 10cc9124, c590b2a9, 154203e9, 6829f..."
1,972da8ab,0.26,1,[8d097b4a]
2,764b2e98,100.00,1,[525134f6]
3,22655375,0.14,7,"[21fd893d, ed2f4471, b327d1e5, ca4e956e, 42067..."
4,f006f6ea,0.32,1,[bbb8412f]
...,...,...,...,...
123,d8d6d1e0,0.93,1,[139361ff]
124,b022aa11,0.20,0,[None]
125,eb5d8f42,0.79,0,[None]
126,1e64c783,1.00,0,[None]


In [32]:
underloaded[underloaded["vm_count"] == 0]

,node_name,cpu_usage_percent,vm_count,vm_ids
10,fe2955c5,0.09,0,[None]
11,bd55f82d,0.09,0,[None]
12,5e87c22e,0.17,0,[None]
13,0861be67,0.04,0,[None]
14,7f3747b5,0.10,0,[None]
31,fb77ceef,0.48,0,[None]
32,41721e4b,1.06,0,[None]
42,a2ffa1e3,0.09,0,[None]
43,7c7d255d,0.04,0,[None]
44,5ede733f,0.08,0,[None]


There are mony occurences in which the node is being underutilised but is not 0 and there are no VMs on them. 
This does not affect host detection but when it comes to vm selection there are no VMs to select and non to migrate so the node should be put to idle

In [31]:
overloaded

,node_name,cpu_usage_percent,vm_count,vm_ids
2,764b2e98,100.00,1,[525134f6]
30,46de937b,91.68,0,[None]
37,5433c3e6,99.95,1,[b3382983]
98,05c5ef00,99.65,1,[06e83870]
109,cbf4a974,99.92,1,[5d14c1f6]


In [ ]:
'''
    For underloaded nodes, we migrate all VMs to other nodes
    For overloaded nodes, we migrate VMs until we are under the upper threshold
'''
def selection_algorithm(overloaded, underloaded, t):
    vms_to_migrate = []

    # Underloaded nodes
    for _, row in underloaded.iterrows():
        if row["vm_count"] == 0:
            continue
        
        for vm_id in row["vm_ids"]:
            vms_to_migrate.append({
            "vm_id": vm_id,
            "source_node": row["node_name"]
            })

    # Overloaded nodes
    for _, row in overloaded.iterrows():
        if row["vm_count"] == 0:
            continue
        
        node_name = row["node_name"] # the overloaded node
        
        # get info of the VM
        vm_info = con.execute("""
            SELECT vm_id, power_clean
            FROM vm_final
            WHERE timestamp = ?
            AND hypervisor_name = ?
        """, [t + "UTC", node_name]).df()

        # Larger vms are migrated first
        vm_info = vm_info.sort_values(by="power_clean", ascending=False)


In [ ]:
for (t,) in timestamps: 
    
    # for each node at time t, extract name, cpu usage, and vm count and vm ids
    node_df_t = con.execute("""
                            SELECT 
                                n.node_name,
                                n.cpu_usage_percent,
                                COUNT(v.vm_id) AS vm_count,
                                LIST(v.vm_id) AS vm_ids
                            FROM nodes_table n
                            LEFT JOIN vm_final v
                                ON n.timestamp = v.timestamp
                                AND n.node_name = v.hypervisor_name
                            WHERE n.timestamp = ?
                            GROUP BY n.node_name, n.cpu_usage_percent
                        """, [t + " UTC"]).df()
    
    # Host Detection
    overloaded = node_df_t[node_df_t["cpu_usage_percent"] > UPPER_THRESHOLD]
    underloaded = node_df_t[node_df_t["cpu_usage_percent"] < LOWER_THRESHOLD]

    # VM Selection

    

In [ ]:
# Check if nodes with no vms have cpu usage to 0